# YouTube QA Bot (LangChain + Gemini + FAISS) 

This project lets you ask questions about a YouTube video by:
1. Downloading the video transcript
2. Splitting it into chunks
3. Embedding the chunks using Google Vertex AI embeddings
4. Storing them in a FAISS vector database
5. Querying the most relevant chunks
6. Generating answers using Gemini (Google Generative AI)

---

## 🚀 Features

- YouTube transcript loading via LangChain
- Text chunking with overlap for better context
- Vector search using FAISS
- Semantic search over video content
- Question answering using Gemini (`gemini-2.5-flash`)
- Strict "context-only" answering (no hallucinations)

---

## 📦 Tech Stack

- LangChain
- Google Gemini API (`ChatGoogleGenerativeAI`)
- Google Vertex AI Embeddings
- FAISS (vector database)
- YouTube Transcript Loader

---

## 1. Import libraries

In [1]:
from dotenv import load_dotenv
import os
import textwrap

from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_vertexai import VertexAIEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

/var/folders/_0/xwb7thqj7s3ddhwq0zdlkknw0000gn/T/ipykernel_6886/1973024199.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import YoutubeLoader


## 2. Load environment variables

In [2]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("api key")

## 3. Initialize embeddings (Vertex AI)

In [3]:
# Make sure you ran:
# gcloud auth application-default login

embeddings = VertexAIEmbeddings(
    model_name="text-embedding-005",
    project="452101803676",   # <-- replace with your project id
    location="us-central1"
)

NameError: name 'GoogleGenerativeAIEmbeddings' is not defined

In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
      model="models/text-embedding-004",
      google_api_key=os.getenv("api-key")
  )


ValidationError: 1 validation error for GoogleGenerativeAIEmbeddings
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'models/text-em... 'google_api_key': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

## 4. Function: Load YouTube transcript + build FAISS DB

In [ ]:
def create_db_from_youtube_video_url(video_url: str):

    loader = YoutubeLoader.from_youtube_url(
        video_url,
        add_video_info=False
    )

    transcript = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=100
    )

    docs = text_splitter.split_documents(transcript)

    db = FAISS.from_documents(docs, embeddings)

    return db

## 5. Function: Ask questions from the video

In [ ]:
def get_response_from_query(db, query: str, k: int = 4):

    docs = db.similarity_search(query, k=k)

    docs_page_content = "\n\n".join([doc.page_content for doc in docs])

    model = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.2
    )

    prompt = ChatPromptTemplate.from_template(
        """
You are a helpful assistant that answers questions about a YouTube video
using only the provided transcript context.

Context:
{docs}

Question:
{question}

Rules:
- Only answer from the transcript
- If the transcript does not contain the answer, say "I don't know"
"""
    )

    chain = prompt | model | StrOutputParser()

    response = chain.invoke({
        "docs": docs_page_content,
        "question": query
    })

    return response, docs

## 6. Load a YouTube video

In [ ]:
video_url = "https://www.youtube.com/watch?v=2P27Ef-LLuQ"

db = create_db_from_youtube_video_url(video_url)

print("Vector DB created successfully!")

## 7. Ask a question

In [ ]:
query = "what are they saying about Impact on Jobs?"

response, docs = get_response_from_query(db, query)

print(textwrap.fill(response, width=80))

# Evaluation


# Gold Set

In [ ]:
import time
import psutil
import json as json_module
from datetime import datetime
from pathlib import Path

GOLD_SET = [
    {
        "query": "How does Sam Altman describe OpenAI's strategy to win against competitors?",
        "relevant_keywords": ["openai", "strategy", "competition", "win", "competitor", "google", "anthropic"],
        "expected_answer_elements": ["strategy, competition, win", "openai, market"]
    },
    {
        "query": "What does Sam Altman say about the future of ChatGPT?",
        "relevant_keywords": ["chatgpt", "future", "product", "feature", "model"],
        "expected_answer_elements": ["chatgpt, future", "product, improve"]
    },
    {
        "query": "What is the reasoning behind OpenAI's large infrastructure investments?",
        "relevant_keywords": ["infrastructure", "investment", "compute", "data center", "build"],
        "expected_answer_elements": ["infrastructure, compute", "investment, build"]
    },
    {
        "query": "What are OpenAI's IPO plans?",
        "relevant_keywords": ["ipo", "public", "stock", "valuation", "2026"],
        "expected_answer_elements": ["ipo, public", "valuation, 2026"]
    },
    {
        "query": "What does Sam Altman say about the impact of AI on jobs?",
        "relevant_keywords": ["jobs", "employment", "work", "impact", "automation"],
        "expected_answer_elements": ["jobs, employment, work", "impact, automation"]
    },
]

# Metric functions

In [ ]:

def precision_at_k(docs, relevant_keywords, k):
    top = docs[:k]
    hits = sum(1 for doc in top if any(kw.lower() in doc.page_content.lower() for kw in relevant_keywords))
    return hits / k if k > 0 else 0.0

def recall_at_k(docs, relevant_keywords, k):
    top = docs[:k]
    found = {kw for doc in top for kw in relevant_keywords if kw.lower() in doc.page_content.lower()}
    return len(found) / max(1, len(relevant_keywords))

def compute_grounding(answer_text, expected_elements):
    if not expected_elements:
        return 1.0
    answer_lower = answer_text.lower()
    found = sum(1 for elem in expected_elements
                if any(kw.strip().lower() in answer_lower for kw in elem.split(",")))
    return found / len(expected_elements)

def compute_elements_coverage(answer_text, expected_elements):
    if not expected_elements:
        return 1.0, [], []
    answer_lower = answer_text.lower()
    found, missing = [], []
    for elem in expected_elements:
        keywords = [kw.strip().lower() for kw in elem.split(",")]
        (found if any(kw in answer_lower for kw in keywords) else missing).append(elem)
    return len(found) / len(expected_elements), found, missing

def compute_faithfulness(answer_text, docs):
    if not docs:
        return 0.0
    answer_lower = answer_text.lower()
    hits = sum(
        1 for doc in docs
        if sum(1 for w in doc.page_content.split()[:80] if len(w) > 5 and w.lower() in answer_lower) >= 3
    )
    return hits / len(docs)

def check_response_format(response_text):
    if not response_text or len(response_text.strip()) < 10:
        return False, "empty response"
    if response_text.strip().lower() in ["i don't know", "i do not know"]:
        return False, "full refusal"
    return True, "ok"

# Pipeline

In [ ]:
def run_evaluation_pipeline(db, gold_item, k=4):
    t0 = time.time()
    response, docs = get_response_from_query(db, gold_item["query"], k=k)
    total_time = time.time() - t0

    prec         = precision_at_k(docs, gold_item["relevant_keywords"], k)
    rec          = recall_at_k(docs, gold_item["relevant_keywords"], k)
    grounding    = compute_grounding(response, gold_item["expected_answer_elements"])
    elem_cov, found_elems, missing_elems = compute_elements_coverage(response, gold_item["expected_answer_elements"])
    faithfulness = compute_faithfulness(response, docs)
    fmt_ok, fmt_note = check_response_format(response)

    return {
        "query":             gold_item["query"],
        "response":          response,
        "docs":              docs,
        "precision_at_k":    round(prec, 3),
        "recall_at_k":       round(rec, 3),
        "grounding_score":   round(grounding, 3),
        "elements_coverage": round(elem_cov, 3),
        "found_elements":    found_elems,
        "missing_elements":  missing_elems,
        "faithfulness":      round(faithfulness, 3),
        "format_ok":         fmt_ok,
        "format_note":       fmt_note,
        "total_time_s":      round(total_time, 3),
        "tokens_in":         len(gold_item["query"].split()),
        "tokens_out":        len(response.split()),
    }

results_eval = []
for item in GOLD_SET:
    print(f"Evaluating: {item['query'][:60]}...")
    results_eval.append(run_evaluation_pipeline(db, item, k=4))

print(f"\nDone: {len(results_eval)} queries evaluated")

# Results table

In [ ]:
print(f"{'Query':<45} | {'Prec@k':>6} | {'Rec@k':>5} | {'Ground':>6} | {'ElemCov':>7} | {'Faith':>5} | {'Fmt':>4} | {'Time':>6}")
print("-" * 108)
for r in results_eval:
    print(f"{r['query'][:45]:<45} | "
          f"{r['precision_at_k']:>6.3f} | "
          f"{r['recall_at_k']:>5.3f} | "
          f"{r['grounding_score']:>6.3f} | "
          f"{r['elements_coverage']:>7.3f} | "
          f"{r['faithfulness']:>5.3f} | "
          f"{'OK' if r['format_ok'] else 'FAIL':>4} | "
          f"{r['total_time_s']:>5.2f}s")
print("-" * 108)
n = len(results_eval)
print(f"{'AVERAGE':<45} | "
      f"{sum(r['precision_at_k']    for r in results_eval)/n:>6.3f} | "
      f"{sum(r['recall_at_k']       for r in results_eval)/n:>5.3f} | "
      f"{sum(r['grounding_score']   for r in results_eval)/n:>6.3f} | "
      f"{sum(r['elements_coverage'] for r in results_eval)/n:>7.3f} | "
      f"{sum(r['faithfulness']      for r in results_eval)/n:>5.3f} | "
      f"{sum(1 for r in results_eval if r['format_ok'])}/{n:<2}   | "
      f"{sum(r['total_time_s']      for r in results_eval)/n:>5.2f}s")

# System metrics

In [ ]:
latencies = [r["total_time_s"] for r in results_eval]
tok_in    = [r["tokens_in"]    for r in results_eval]
tok_out   = [r["tokens_out"]   for r in results_eval]
n = len(results_eval)

print("=" * 55)
print("SYSTEM METRICS")
print("=" * 55)
print(f"\n-- Latency -----------------------------------------")
print(f"  Mean total latency   : {sum(latencies)/n:.2f}s")
print(f"  Min / Max            : {min(latencies):.2f}s / {max(latencies):.2f}s")
print(f"  Queries / minute     : {60 / (sum(latencies)/n):.2f}")
print(f"\n-- Tokens ------------------------------------------")
print(f"  Mean tokens in       : {sum(tok_in)/n:.1f}")
print(f"  Mean tokens out      : {sum(tok_out)/n:.1f}")
print(f"  Total tokens in      : {sum(tok_in)}")
print(f"  Total tokens out     : {sum(tok_out)}")
print(f"\n-- Cost --------------------------------------------")
print(f"  Model                : gemini-2.5-flash (Google API)")
print(f"  Note                 : cost based on tokens above, see ai.google.dev/pricing")
print(f"\n-- Per-query breakdown -----------------------------")
print(f"{'Query':<45} | {'Time':>6} | {'Tok_in':>6} | {'Tok_out':>7}")
print("-" * 72)
for r in results_eval:
    print(f"{r['query'][:45]:<45} | {r['total_time_s']:>5.2f}s | {r['tokens_in']:>6} | {r['tokens_out']:>7}")

# Manual evaluation

In [ ]:
RUBRIC = {
    "relevance":    "Is the answer relevant to the question and the video?",
    "correctness":  "Is it factually correct based on what Sam Altman said?",
    "faithfulness": "Is it grounded in the transcript, no hallucinations?",
    "completeness": "Does it cover the key points expected?",
    "clarity":      "Is it clearly written and easy to understand?",
}

for i, r in enumerate(results_eval):
    print(f"\n{'='*70}")
    print(f"[{i+1}/{len(results_eval)}] QUERY: {r['query']}")
    print(f"\nTop retrieved chunks:")
    for j, doc in enumerate(r["docs"][:3]):
        print(f"  [{j+1}] {doc.page_content[:120]}...")
    print(f"\nGenerated answer:")
    print(f"  {r['response'][:300]}{'...' if len(r['response']) > 300 else ''}")
    print(f"\nAuto: Prec@k={r['precision_at_k']:.3f} | Rec@k={r['recall_at_k']:.3f} | Ground={r['grounding_score']:.3f} | Faith={r['faithfulness']:.3f} | Fmt={r['format_note']}")
    print(f"Missing elements: {r['missing_elements']}")
    print(f"\n{'-'*70}")
    print("SCORING GRID (0=bad, 1=acceptable, 2=good):")
    print(f"  {'Criterion':<15} | {'Score':>5} | Description")
    print(f"  {'-'*60}")
    for criterion, description in RUBRIC.items():
        print(f"  {criterion:<15} | {'?/2':>5} | {description}")

# Regression testing

In [ ]:
REGRESSION_SAVE_PATH = "./regression_baseline.json"

def save_or_compare_regression(results_eval, save_path):
    path = Path(save_path)
    snapshot = {
        "timestamp": datetime.now().isoformat(),
        "video_url": video_url,
        "scores": {
            r["query"]: {
                "precision_at_k":    r["precision_at_k"],
                "recall_at_k":       r["recall_at_k"],
                "grounding_score":   r["grounding_score"],
                "elements_coverage": r["elements_coverage"],
                "faithfulness":      r["faithfulness"],
            }
            for r in results_eval
        }
    }
    if not path.exists():
        with open(path, "w") as f:
            json_module.dump(snapshot, f, indent=2)
        print(f"Baseline saved ({len(results_eval)} queries). Re-run after a config change to detect regressions.")
        return
    with open(path) as f:
        baseline = json_module.load(f)
    print(f"Baseline : {baseline['timestamp']}")
    print(f"Current  : {snapshot['timestamp']}")
    THRESHOLD = 0.05
    metrics = ["precision_at_k", "recall_at_k", "grounding_score", "elements_coverage", "faithfulness"]
    regressions = []
    print(f"\n{'Query':<45} | {'Metric':<18} | {'Before':>6} | {'After':>6} | {'Delta':>7}")
    print("-" * 90)
    for r in results_eval:
        q = r["query"]
        if q not in baseline["scores"]:
            continue
        for metric in metrics:
            before = baseline["scores"][q].get(metric, 0.0)
            after  = snapshot["scores"][q].get(metric, 0.0)
            delta  = after - before
            if delta < -THRESHOLD:
                regressions.append((q, metric, before, after))
            if abs(delta) > 0.01:
                flag = " <-- REGRESSION" if delta < -THRESHOLD else ""
                print(f"{q[:45]:<45} | {metric:<18} | {before:>6.3f} | {after:>6.3f} | {delta:>+7.3f}{flag}")
    print("-" * 90)
    print(f"\n{len(regressions)} regression(s) detected." if regressions else "\nNo regressions detected.")

save_or_compare_regression(results_eval, REGRESSION_SAVE_PATH)